# Cross-country comparison — Week 0 Task 3

Reads `data/ethiopia_clean.csv` … `data/nigeria_clean.csv` produced in Task 2.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "notebooks"))

from compare_pipeline import (
    agg_table_precip,
    agg_table_t2m,
    anova_or_kruskal_t2m,
    bar_by_country,
    extreme_heat_days_by_year,
    load_all_clean,
    max_dry_days_by_year_country,
    monthly_t2m_overlay,
    precip_boxplots,
    vulnerability_rank_summary,
)

sns.set_theme(style="whitegrid")
df = load_all_clean(ROOT)
display(df["Country"].value_counts())
df.head()


## Monthly average T2M — all countries + summary table


In [ ]:
fig = monthly_t2m_overlay(df)
plt.tight_layout()
plt.show()
display(agg_table_t2m(df))


## Daily PRECTOTCORR — side-by-side boxplots + summary table


In [ ]:
fig = precip_boxplots(df)
plt.show()
display(agg_table_precip(df))


## Extreme heat (T2M_MAX > 35°C) & consecutive dry days (< 1 mm/day)


In [ ]:
heat = extreme_heat_days_by_year(df)
drought = max_dry_days_by_year_country(df)
display(heat.tail())
display(drought.tail())

fig = bar_by_country(heat, "heat_days_35", "Days per year with T2M_MAX > 35°C")
plt.show()
fig = bar_by_country(
    drought,
    "max_consecutive_dry_days",
    "Max consecutive dry days (PRECTOTCORR < 1 mm) per year",
)
plt.show()


## Statistical test on T2M across countries


In [ ]:
name, stat, pval, obj = anova_or_kruskal_t2m(df)
print(f"{name}: statistic={stat:.4g}, p-value={pval:.4g}")
if pval < 0.05:
    print("Small p-value: strong evidence that at least one country differs in mean rank / location.")
else:
    print("Large p-value: no strong evidence against equal distributions — still compare effect sizes.")


### Notes on testing

**ANOVA** if roughly normal homogeneous groups; otherwise **Kruskal–Wallis** when variances diverge heavily.


In [ ]:
rank_df = vulnerability_rank_summary(df, heat, drought)
display(rank_df)


## COP32 framing — five bullets (edit after running on official data)

- **Which country warms fastest?** Compare slopes on the overlay — tie to anomalies vs a baseline if extended.
- **Most unstable precipitation:** highest daily spread (std/boxplots) plus dry spell maxima.
- **What extremes imply:** heat-day counts × dry spells show compound hazard for agriculture/water.
- **Ethiopia vs neighbors:** read Ethiopia’s row vs peers in the tables and figures.
- **Who to champion for finance:** pick the country with converging high stress in the ranking + narrative for **adaptation finance**, **loss & damage**, and **early warning** — justify with this table.
